In [1]:
"""读取parquet文件"""
import pandas as pd

# 读取 parquet 文件
input_parquet = "/home/jiaqizheng/my_dataset/Tool_use/grpo/rl_train.parquet"
df = pd.read_parquet(input_parquet)

# 打印前几行查看内容

print(df.head())
df.to_json("/home/jiaqizheng/my_dataset/Tool_use/grpo/rl_train.json", orient="records",indent=4, force_ascii=False)




     data_source                                             prompt ability  \
0  mail/tool_use  [{'attachments': None, 'content': '你是一个智能邮件助手，...    math   
1  mail/tool_use  [{'attachments': None, 'content': '你是一个智能邮件助手，...    math   
2  mail/tool_use  [{'attachments': None, 'content': '你是一个智能邮件助手，...    math   
3  mail/tool_use  [{'attachments': None, 'content': '你是一个智能邮件助手，...    math   
4  mail/tool_use  [{'attachments': None, 'content': '你是一个智能邮件助手，...    math   

                                        reward_model       extra_info  
0  {'ground_truth': '<tool_call>
{"name": "edit_m...  {'index': 1813}  
1  {'ground_truth': '<tool_call>
{"name": "edit_m...  {'index': 3542}  
2  {'ground_truth': '<tool_call>
{"name": "search...  {'index': 3103}  
3  {'ground_truth': '<tool_call>
{"name": "search...  {'index': 1344}  
4  {'ground_truth': '<tool_call>
{"name": "search...  {'index': 2825}  


In [6]:
"""把 SFT 数据集格式转成 Verl 的 GRPO 数据集格式"""
import json
from pathlib import Path

sft_json = "/home/jiaqizheng/my_dataset/Tool_use/sft/805测试前1000条.json"
grpo_json = "/home/jiaqizheng/my_dataset/Tool_use/grpo/grpo_1000_805.json"

with open(sft_json, "r", encoding="utf-8") as f:
    sft_data = json.load(f)

grpo_data = []

def wrap_tool_call(tool_call_str: str) -> str:
    # 注意这里不要写 <\/tool_call>，也不要把 / 转义
    return f"<tool_call>\n{tool_call_str}\n</tool_call>"

for idx, item in enumerate(sft_data):
    msgs = item["messages"]
    last = msgs[-1]

    # 有些样本可能没有 tool_calls，跳过或按需处理
    tc_list = last.get("tool_calls") or []
    if not tc_list:
        continue

    fn = tc_list[0]["function"]
    name = fn["name"]
    args = fn["arguments"]  # 可能是 str 也可能是 dict

    # 统一成可序列化的对象
    if isinstance(args, str):
        try:
            args_obj = json.loads(args)
        except json.JSONDecodeError:
            # 如果确实就是字符串，按字符串塞进去
            args_obj = args
    else:
        args_obj = args

    tool_call_obj = {"name": name, "arguments": args_obj}
    tool_call_str = json.dumps(tool_call_obj, ensure_ascii=False, separators=(",", ":"))

    grpo_entry = {
        "data_source": "mail/tool_use",   # 不要写成 mail\/tool_use
        "prompt": msgs[:-1],
        "ability": "mail",
        "reward_model": {
            "ground_truth": wrap_tool_call(tool_call_str),
            "style": "rule"
        },
        "extra_info": {
            "index": idx
        }
    }
    grpo_data.append(grpo_entry)

Path(grpo_json).parent.mkdir(parents=True, exist_ok=True)
with open(grpo_json, "w", encoding="utf-8") as f:
    json.dump(grpo_data, f, ensure_ascii=False, indent=4)


In [5]:
import json
inputjson = "/home/jiaqizheng/my_dataset/Tool_use/sft/805测试前1000条.json"
with open(inputjson, 'r', encoding='utf-8') as f:
    all_items = json.load(f)

    all_items = all_items[:1000]
    print(len(all_items))
with open(inputjson, 'w', encoding='utf-8') as f:
    json.dump(all_items, f, ensure_ascii=False, indent=4)

1000


In [7]:
"""json文件顺序拆分为训练集和测试集, 转为parquet格式"""
import pandas as pd

def split_json_to_parquet_ordered(json_path, train_parquet_path, test_parquet_path, test_frac=0.2):
    df = pd.read_json(json_path, lines=False)
    n = len(df)
    split = int(n * (1 - test_frac))
    df.iloc[:split].to_parquet(train_parquet_path)
    df.iloc[split:].to_parquet(test_parquet_path)

if __name__ == '__main__':
    json_path = '/home/jiaqizheng/my_dataset/Tool_use/grpo/grpo_1000_805.json'
    train_parquet_path = '/home/jiaqizheng/my_dataset/Tool_use/grpo/grpo_train_805.parquet'
    test_parquet_path = '/home/jiaqizheng/my_dataset/Tool_use/grpo/grpo_test_805.parquet'
    split_json_to_parquet_ordered(json_path, train_parquet_path, test_parquet_path, test_frac=0.2)


In [ ]:
"""读取parquet文件, 检查 Token 数量"""
from transformers import AutoTokenizer
import pandas as pd

# 这里是本地模型目录，而不是 parquet 文件路径
tok = AutoTokenizer.from_pretrained(
    "/home/jiaqizheng/my_models/Qwen3-1.7B",
    local_files_only=True
)

df = pd.read_parquet("/home/jiaqizheng/my_dataset/Tool_use/grpo/grpo_train_805.parquet")

lengths = df["prompt"].apply(lambda x: len(tok(x).input_ids))
print(lengths.describe())


HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/home/jiaqizheng/my_dataset/Tool_use/grpo/grpo_train_805.parquet'. Use `repo_type` argument if needed.

In [23]:
import pandas as pd
import json

# 指定 JSON 文件路径
json_file_path = "/home/jiaqizheng/my_dataset/Tool_use/tool_use_test_v4.json"

# 方法一：如果 JSON 文件是一个“数组”（即最外层是一个列表，每个元素是一个字典）
df = pd.read_json(json_file_path)


df_filtered = df[df.iloc[:, 4] == 0]
# 显示前几行查看结果
# df.iloc[4:7]
df_filtered

,messages,ground_truth,tooluse_sft_lora_v4,tooluse_sft_full_v4,lora得分_v4,lora原因_v4,full得分_v4,full原因_v4
1,"[{""role"": ""system"", ""content"": ""你是一个智能邮件助手，可以使...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...",0,"虽然模型预测的工具名称与参数搭配都准确，但必填字段 'fromAccount' 的值是 ""A...",0,工具名相同，参数名正确，语义对齐，但 fromAccount 是必填字段且值不正确。
4,"[{""role"": ""system"", ""content"": ""你是一个智能邮件助手，可以使...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...",0,The tool name is correct and the XML structure...,0,fromAccount 字段错误，必填字段值不匹配。
7,"[{""role"": ""system"", ""content"": ""你是一个智能邮件助手，可以使...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...",0,工具名一致且参数合法，但必填字段 fromAccount 的值不正确。,0,工具名称一致，所有必填字段名称也都正确。虽然 'fromAccount' 的值不同，但在此工...
10,"[{""role"": ""system"", ""content"": ""你是一个智能邮件助手，可以使...","<tool_call>\n{""name"": ""search_data"", ""argument...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...","<tool_call>\n{""name"": ""search_data"", ""argument...",0,"工具名称 ""name"" 不一致，ground truth 是 ""search_data""，而...",1,字段一致，语义对齐。虽然预测中描述略有不同，但信息检索意图相同。
11,"[{""role"": ""system"", ""content"": ""你是一个智能邮件助手，可以使...","<tool_call>\n{""name"": ""get_mail"", ""arguments"":...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...","<tool_call>\n{""name"": ""reply_mail"", ""arguments...",0,工具名不一致：预测结果为“send_mail”，而标准答案为“get_mail”。,0,工具调用的名称不匹配。模型预测的工具是 `reply_mail`，而标准答案的工具是 `ge...
12,"[{""role"": ""system"", ""content"": ""你是一个智能邮件助手，可以使...","<tool_call>\n{""name"": ""reply_mail"", ""arguments...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...","<tool_call>\n{""name"": ""reply_mail"", ""arguments...",0,"工具名不匹配，标准答案使用的是""reply_mail""，而模型预测使用了""send_mail...",0,"The tool name is correct, and the XML structur..."
14,"[{""role"": ""system"", ""content"": ""你是一个智能邮件助手，可以使...","<tool_call>\n{""name"": ""search_data"", ""argument...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...","<tool_call>\n{""name"": ""search_data"", ""argument...",0,工具名不一致，预期为 `search_data` 而模型提供为 `send_mail`，因此...,1,工具名和参数名称均匹配，且参数语义一致。
15,"[{""role"": ""system"", ""content"": ""你是一个智能邮件助手，可以使...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...","<tool_call>\n{""name"": ""send_mail"", ""arguments""...",0,"The tool name ""send_mail"" matches the ground t...",0,The 'fromAccount' field in the predicted resul...
18,"[{""role"": ""system"", ""content"": ""你是一个智能邮件助手，可以使...","<tool_call>\n{""name"": ""save_draft"", ""arguments...","<tool_call>\n{""name"": ""save_draft"", ""arguments...","<tool_call>\n{""name"": ""save_draft"", ""arguments...",0,"The tool name matches, the XML structure is va...",0,"Although the tool names match, the 'fromAccoun..."
21,"[{""role"": ""system"", ""content"": ""你是一个智能邮件助手，可以使...","<tool_call>\n{""name"": ""search_data"", ""argument...","<tool_call>\n{""name"": ""forward_mail"", ""argumen...","<tool_call>\n{""name"": ""search_data"", ""argument...",0,"工具名不一致。Ground Truth 使用了 ""search_data""，而 Predic...",1,字段完全一致，语义对齐。虽然 arguments 的格式不同但语义相同。
